# **Lab project 3: Domain Alignment with GANs for robust classifier generalization**

**Pablo Martínez Olmos, Vanessa Gómez Verdejo,  Alejandro Lancho Serrano, Emilio Parrado Hernández**

Departamento de Teoría de la Señal y Comunicaciones

**Universidad Carlos III de Madrid**

<img src='https://upload.wikimedia.org/wikipedia/commons/8/88/Acronimo_nombre3l.jpg' width=300 />


In this notebook, you will deploy domain alignment using a **CycleGAN-style architecture** on **tabular data**, followed by downstream evaluation with classical classifiers.

The goal is to show how data coming from two different domains (A and B) can be aligned *before* training a classifier, improving generalization under domain shift.

---

## 1. Problem setup

We consider a **binary classification problem on tabular data**. The original dataset is split into two domains:

- **Domain A**: clean data (reference domain)
- **Domain B**: a synthetically shifted version of A, obtained via
  - deterministic smooth nonlinear transformations
  - additive noise and feature corruption

The task is to **undo the domain shift** in B using a GAN-based alignment model.


## 2. Imports and reproducibility

We load a binary classification dataset from the UCI repository (Breast Cancer Wisconsin Diagnostic).

The dataset consists of:
- numeric features only
- binary target labels

In [2]:
# pip install -q scikit-learn pandas

import pandas as pd
from sklearn.datasets import load_breast_cancer

ds = load_breast_cancer(as_frame=True)
X = ds.data
y = ds.target  # 0/1 labels

print("X shape:", X.shape)
print("y value counts:\n", y.value_counts())
print(X.head())

X shape: (569, 30)
y value counts:
 target
1    357
0    212
Name: count, dtype: int64
   mean radius  mean texture  mean perimeter  mean area  mean smoothness  \
0        17.99         10.38          122.80     1001.0          0.11840   
1        20.57         17.77          132.90     1326.0          0.08474   
2        19.69         21.25          130.00     1203.0          0.10960   
3        11.42         20.38           77.58      386.1          0.14250   
4        20.29         14.34          135.10     1297.0          0.10030   

   mean compactness  mean concavity  mean concave points  mean symmetry  \
0           0.27760          0.3001              0.14710         0.2419   
1           0.07864          0.0869              0.07017         0.1812   
2           0.15990          0.1974              0.12790         0.2069   
3           0.28390          0.2414              0.10520         0.2597   
4           0.13280          0.1980              0.10430         0.1809   

   me

## 4. Domain construction (A vs B)

The dataset is randomly split into two equal parts:

- **Set A**: used as-is
- **Set B**: transformed to simulate a domain shift

All features are standardized using statistics computed **only from A**.

### 4.1 Deterministic smooth transformation

Let $x \in \mathbb{R}^d$ be a standardized sample from domain \(B\).
We apply a deterministic, smooth feature warp
$$
x^{\text{det}} = T_{\text{det}}(x),
$$
defined componentwise as
$$
x^{\text{det}}_j
= x_j
+ \alpha_j \tanh(\beta_j x_j + \phi_j)
+ 0.05\,s\,\tanh(x_{j-1}),
$$
with wrap-around indexing for $x_{j-1}$. The parameters are fixed per feature:
$$
\alpha_j = s\left(0.15 + 0.10\sin(0.7j)\right),\quad
\beta_j  = 0.8 + 0.3\cos(0.5j),\quad
\phi_j   = 0.2\sin(1.3j),
$$

where $s$ is the warping strength. This introduces a structured, smooth, nonlinear domain shift.


### 4.2 Stochastic corruption

After the deterministic warp, we apply stochastic perturbations:
$$
x^{B} = \text{Dropout}\big(m \odot x^{\text{det}} + b + \epsilon\big),
$$
where
$$
m_j \sim \mathcal{N}(1,\sigma_{\text{scale}}^2),\quad
b_j \sim \mathcal{N}(0,\sigma_{\text{shift}}^2),\quad
\epsilon_{ij} \sim \mathcal{N}(0,\sigma_{\text{gauss}}^2).
$$

Feature dropout is applied independently with probability $p_{\text{drop}}$. This combination of **systematic nonlinear distortion** and **random corruption** produces a challenging but realistic domain shift.



### Domain shift parameters (summary)

| Component | Parameter | Value |
|---------|-----------|-------|
| Split | Train/test ratio | 50% A / 50% B |
| Standardization | Statistics source | Set A only |
| Deterministic warp | Strength $s$ | 0.8 |
| Deterministic warp | $\alpha_j$ | $0.8(0.15 + 0.10\sin(0.7j))$ |
| Deterministic warp | $\beta_j$ | $0.8 + 0.3\cos(0.5j)$ |
| Deterministic warp | $\phi_j$ | $0.2\sin(1.3j)$ |
| Cross-feature term | Neighbor weight | $0.05\,s = 0.04$ |
| Gaussian noise | $\sigma_{\text{gauss}}$ | 0.55 |
| Feature scaling | $\sigma_{\text{scale}}$ | 0.80 |
| Feature shifting | $\sigma_{\text{shift}}$ | 0.80 |
| Feature dropout | $p_{\text{drop}}$ | 0.60 |

---


In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------------------
# 1) Split into Set A / Set B
# ---------------------------
XA_df, XB_df, yA_s, yB_s = train_test_split(
    X, y, test_size=0.5, random_state=42, stratify=y
)

# ---------------------------
# 2) Standardize using ONLY Set A stats
# ---------------------------
scaler = StandardScaler()
XA = scaler.fit_transform(XA_df.values.astype(np.float32))
XB = scaler.transform(XB_df.values.astype(np.float32))

rng = np.random.default_rng(42)

# ---------------------------
# 3) Deterministic smooth transformation (domain B)
# ---------------------------
def smooth_domain_transform(XB_std: np.ndarray, strength: float = 0.8) -> np.ndarray:
    d = XB_std.shape[1]
    X = XB_std

    i = np.arange(d, dtype=np.float32)
    alpha = strength * (0.15 + 0.10 * np.sin(0.7 * i))
    beta  = 0.8 + 0.3 * np.cos(0.5 * i)
    phase = 0.2 * np.sin(1.3 * i)

    alpha = alpha[None, :]
    beta  = beta[None, :]
    phase = phase[None, :]

    X_warp = X + alpha * np.tanh(beta * X + phase)

    # cross-feature smooth interaction
    X_roll = np.roll(X, shift=1, axis=1)
    X_warp += (strength * 0.05) * np.tanh(X_roll)

    return X_warp.astype(np.float32)

XB_det = smooth_domain_transform(XB)

# ---------------------------
# 4) Stochastic corruption (domain B)
# ---------------------------
sigma_gauss = 0.55
scale_jitter = 0.80
shift_jitter = 0.80
p_dropout = 0.60

gauss = rng.normal(0.0, sigma_gauss, size=XB_det.shape).astype(np.float32)
mult  = rng.normal(1.0, scale_jitter, size=(1, XB_det.shape[1])).astype(np.float32)
shift = rng.normal(0.0, shift_jitter, size=(1, XB_det.shape[1])).astype(np.float32)

XB_noisy = XB_det * mult + shift + gauss

if p_dropout > 0:
    mask = rng.random(XB_noisy.shape) < p_dropout
    XB_noisy[mask] = 0.0

# Labels unchanged
yA = yA_s.values.astype(np.int64)
yB = yB_s.values.astype(np.int64)

print("Set A:", XA.shape, "labels:", np.bincount(yA))
print("Set B (det + noisy):", XB_noisy.shape, "labels:", np.bincount(yB))


Set A: (284, 30) labels: [106 178]
Set B (det + noisy): (285, 30) labels: [106 179]


## 5. Visualizing domain shift

Analyze statistically both domains and visualize them with TSNE. Evaluate the sample [Wasserstein_distance](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wasserstein_distance.html) between both domains.

In [6]:
# COMPLETE

## 6. Baseline classifiers (no alignment)

We evaluate two standard classifiers:

- Logistic Regression (linear baseline)
- Gradient Boosting (nonlinear baseline)

### Experimental settings

1. **Train on A only** → test on A and B
2. **Train on B only** → test on A and B


In [8]:
# COMPLETE

## 7. ROC analysis (before alignment)

Receiver Operating Characteristic (ROC) curves are plotted for both domains.

Expected outcome:
- Good performance on A
- Significant degradation on B due to domain shift

In [9]:
# COMPLETE

## 8. GAN-based domain alignment

We implement a **CycleGAN-style architecture for tabular data**:

- Generators:
  - G_AB: maps A → B
  - G_BA: maps B → A
- Discriminators:
  - D_A: real vs fake A
  - D_B: real vs fake B

During training, we log and visualize:

- discriminator losses
- generator adversarial loss
- cycle consistency loss
- identity loss

This helps diagnose instability or mode collapse.

In [11]:
# COMPLETE

## 10. Visualize the aligned domain

Use TSNE and evaluate the Wasserstein distance between A and B (aligned with CycleGAN)

In [12]:
# COMPLETE

## 11. Classification after alignment

Repeat the classification experiments, now considering the aligned B domain. Analyze your results.

In [13]:
# COMPLETE